# 工具呼叫


**說明**

本教材參考 Udacity 的 AI Agents with LangChain and LangGraph 課程。此程式為 AI 代理程式加入工具呼叫能力，讓它可以動態地與外部函式互動。


## 0. 匯入必要的套件


In [1]:
import os
import requests
import datetime
import inspect
import json
from typing import (
    TypedDict, 
    List, Dict, Literal, 
    Callable, Optional, Any, 
    get_type_hints
)
from openai import OpenAI
from openai.types.chat.chat_completion_message import ChatCompletionMessage
from openai.types.chat.chat_completion_message_tool_call import ChatCompletionMessageToolCall

## 1. 如何使用 OpenAI 用戶端


若要連接 OpenAI，請先將你的 API 金鑰設定為名為 `OPENAI_API_KEY` 的系統環境變數。

接著讀取該環境變數來建立 OpenAI 用戶端。
```python
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
```


In [2]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

## 2. 複習：記憶與函式


前面我們將記憶功能與自訂 Python 函式結合，建立出工具呼叫的完整循環，讓 LLM 可以與外部世界互動。


In [3]:
class Memory:
    def __init__(self):
        self._messages: List[Dict[str, str]] = []
    
    def add_message(self, 
                    role: Literal['user', 'system', 'assistant', 'tool'], 
                    content: str,
                    tool_calls: dict=dict(),
                    tool_call_id=None)-> None:

        message = {
            "role": role,
            "content": content,
            "tool_calls": tool_calls,
        }

        if role == "tool":
            message = {
                "role": role,
                "content": content,
                "tool_call_id": tool_call_id,
            }

        self._messages.append(message)

    def get_messages(self) -> List[Dict[str, str]]:
        return self._messages

    def last_message(self) -> None:
        if self._messages:
            return self._messages[-1]

    def reset(self) -> None:
        self._messages = []

In [4]:
def chat_with_tools(user_question:str=None, 
                    memory:Memory=None, 
                    model:str="gpt-4o-mini", 
                    temperature=0.0, 
                    tools=None)-> str:
    messages = [{"role": "user", "content": user_question}]
    if memory:
        if user_question:
            memory.add_message(role="user", content=user_question)
        messages = memory.get_messages()        
    
    response = client.chat.completions.create(
        model = model,
        temperature = temperature,
        messages = messages,
        tools=tools,
    )
    
    ai_message = str(response.choices[0].message.content)
    tool_calls = response.choices[0].message.tool_calls
    
    if memory:
        memory.add_message(role="assistant", content=ai_message, tool_calls=tool_calls)
    
    return ai_message

In [5]:
def power(base:float, exponent:float):
    """Exponentatiation: base to the power of exponent"""
    
    return base ** exponent

In [8]:
tools = [{
    "type": "function",
    "function": {
        "name": "power",
        "description": "Exponentatiation: base to the power of exponent",
        "parameters": {
            "type": "object",
            "properties": {
                "base": {"type": "number"},
                "exponent": {"type": "number"}
            },
            "required": ["base", "exponent"],
            "additionalProperties": False
        },
        "strict": True
    }
}]

In [9]:
# Instantiate memory and start with the system prompt
memory = Memory()
memory.add_message(role="system", content="You're a helpful assitant")

# Call the LLM with a question that needs a tool
ai_message = chat_with_tools(
    "2 to the power of -5?",
    model="gpt-3.5-turbo",
    tools=tools,
    memory=memory,
)

# Get the arguments from the tool_calls object and call the actual defined function
args = json.loads(memory.last_message()['tool_calls'][0].function.arguments)
result = power(args["base"], args["exponent"])

# Extract the tool_call_id and feed the LLM with the result from the function 
tool_call_id = memory.last_message()['tool_calls'][0].id
memory.add_message(role="tool", content=str(result), tool_call_id=tool_call_id)
ai_message = chat_with_tools(
    model="gpt-3.5-turbo",
    tools=tools,
    memory=memory,
)

In [10]:
memory.get_messages()

[{'role': 'system', 'content': "You're a helpful assitant", 'tool_calls': {}},
 {'role': 'user', 'content': '2 to the power of -5?', 'tool_calls': {}},
 {'role': 'assistant',
  'content': 'None',
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_x96YLv3qBzRlMNjoxwM1Ykco', function=Function(arguments='{"base":2,"exponent":-5}', name='power'), type='function')]},
 {'role': 'tool',
  'content': '0.03125',
  'tool_call_id': 'call_x96YLv3qBzRlMNjoxwM1Ykco'},
 {'role': 'assistant',
  'content': '2 to the power of -5 is equal to 0.03125.',
  'tool_calls': None}]

## 3. 建立工具抽象層


手動呼叫工具很容易出錯。萬一沒有傳入正確型別，或是在 JSON schema 中漏掉必要欄位

可以建立一個抽象層，讓建立工具與呼叫工具變得更簡單。



你的類別至少應該包含以下方法：

- `__init__()`：接收函式，並加入用來擷取文件字串、參數與型別的邏輯。
- `dict()`：回傳 JSON schema。
- `__call__()`：讓建立出的物件本身可以被呼叫。

範例：
```python
class Tool:
    def __init__(self, func:Callable):
        self.func = func
    
    def dict(self):
        pass

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)  

def my_func(arg1:int)->str:
    return "ok"

my_tool = Tool(my_func)
my_tool(arg1=1)
```



為了理解如何解析函式、取得文件字串、參數與型別，熟悉以下 Python 方法很重要：

- `typing.get_type_hints()`
- `inspect.signature()`


In [11]:
class Tool:
    def __init__(self, func:Callable):
        self.func = func
        self.name = func.__name__
        self.description = func.__doc__
        self.argument_types_map = get_type_hints(func)
        self.signature = inspect.signature(func)
        self.arguments = [
            {
                "name": key, 
                "type": self._infer_json_schema_type(value),
                "required": param.default == inspect.Parameter.empty
            } 
            for key, value in self.argument_types_map.items()
            if (param := self.signature.parameters.get(key))
        ]

    def dict(self):
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parallel_tool_calls": False,
                "parameters": {
                    "type": "object",
                    "properties": {
                        argument["name"]: {
                            "type": argument["type"],
                        }
                        for argument in self.arguments
                    },
                    "required": [
                        argument["name"] 
                        for argument in self.arguments 
                        if argument["required"]
                    ],
                    "additionalProperties": False,
                },
                "strict": True
            }
        }

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)
    
    def _infer_json_schema_type(self, arg_type: Any) -> str:
        if arg_type == bool:
            return "boolean"
        elif arg_type == int:
            return "integer"
        elif arg_type == float:
            return "number"
        elif arg_type == str:
            return "string"
        elif arg_type == list:
            return "array"
        elif arg_type == dict:
            return "object"
        elif arg_type is None:
            return "null"
        elif arg_type == datetime.date or arg_type == datetime.datetime:
            return "string"  # JSON Schema treats dates as strings
        else:
            return "string"  # Default to string if type is unknown

In [12]:
power_tool = Tool(power)

In [13]:
power_tool.dict()

{'type': 'function',
 'function': {'name': 'power',
  'description': 'Exponentatiation: base to the power of exponent',
  'parallel_tool_calls': False,
  'parameters': {'type': 'object',
   'properties': {'base': {'type': 'number'}, 'exponent': {'type': 'number'}},
   'required': ['base', 'exponent'],
   'additionalProperties': False},
  'strict': True}}

In [14]:
power_tool(2,3)

8

## 4. 更新 Agent 類別


強化代理程式的邏輯，讓它可以處理使用者查詢，並動態地與外部工具互動。目標是調整代理程式處理使用者訊息、產生回應，以及在必要時呼叫工具的方式。

**目標**

修改以下工作的邏輯：

- 處理使用者輸入：代理程式應記錄並管理對話歷史。
- 產生回應：代理程式會根據先前訊息，使用語言模型產生回覆。
- 判斷何時需要工具：如果完成請求需要工具，代理程式應偵測到這件事並觸發適當函式。
- 處理工具執行與回應：代理程式應執行工具、取得輸出，並將結果整合回對話。

**步驟**

- 更新邏輯，根據 AI 產生的回應判斷是否需要呼叫工具。
- 如果需要工具，使用正確參數執行工具。
- 將工具結果加入對話，讓 AI 可以利用額外資訊修正或補充回應。
- 確保代理程式可以遞迴處理多個工具呼叫；也就是說，如果第一次工具呼叫後的回應又建議使用另一個工具，代理程式也能正確處理。

**注意事項**

- 思考代理程式如何判斷何時要呼叫工具。
- 確保代理程式正確儲存工具回應，才能自然地延續對話。
- 處理在產生最終回覆前，可能需要依序使用多個工具的情況。


In [15]:
class Agent:
    """A tool-calling AI Agent"""

    def __init__(
        self,
        name:str = "Agent", 
        role:str = "Personal Assistant",
        instructions:str = "Help users with any question",
        model:str = "gpt-4o-mini",
        temperature:float = 0.0,
        tools:List[Tool] = [],
    ):
        self.name = name
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature
        self.memory = Memory()
        self.memory.add_message(
            role="system",
            content=f"You're an AI Agent, your role is {self.role}, " 
                    f"and you need to {self.instructions}",
        )

        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

        self.tools = tools
        self.tool_map = {t.name:t for t in tools}
        self.openai_tools = [t.dict() for t in self.tools] if self.tools else None

    def invoke(self, user_message: str) -> str:
        self.memory.add_message(
            role="user",
            content=user_message,
        )

        ai_message = self._get_completion(
            messages = self.memory.get_messages(),
        )

        tool_calls = ai_message.tool_calls
        self.memory.add_message(
            role="assistant",
            content=ai_message.content,
            tool_calls=tool_calls,
        )

        if tool_calls:
            self._call_tools(tool_calls)
            
        return self.memory.last_message()

    def _call_tools(self, tool_calls:List[ChatCompletionMessageToolCall]):
        for t in tool_calls:
            tool_call_id = t.id
            function_name = t.function.name
            args = json.loads(t.function.arguments)
            callable_tool = self.tool_map[function_name]
            result = callable_tool(**args)
            self.memory.add_message(
                role="tool", 
                content=str(result), 
                tool_call_id=tool_call_id
            )

        ai_message = self._get_completion(
            messages = self.memory.get_messages(),
        )

        tool_calls = ai_message.tool_calls

        self.memory.add_message(
            role="assistant",
            content=ai_message.content,
            tool_calls=tool_calls,
        )

        if tool_calls:
            self._call_tools(tool_calls)


    def _get_completion(self, messages:List[Dict])-> ChatCompletionMessage:
        response = self.client.chat.completions.create(
            model=self.model,
            temperature=self.temperature,
            messages=messages,
            tools=self.openai_tools,
        )
        
        return response.choices[0].message


## 5. 建立一些代理程式並試著玩看看


建立一些具備工具的特定用途代理程式，呼叫它們，並檢查它們的記憶內容。


In [16]:
agent = Agent(
    tools=[Tool(power)]
)

In [17]:
agent.invoke("What is 10 + 5?")

{'role': 'assistant', 'content': '10 + 5 equals 15.', 'tool_calls': None}

In [17]:
agent.memory.get_messages()

[{'role': 'system',
  'content': "You're an AI Agent, your role is Personal Assistant, and you need to Help users with any question",
  'tool_calls': {}},
 {'role': 'user', 'content': 'What is 10 + 5?', 'tool_calls': {}},
 {'role': 'assistant', 'content': '10 + 5 equals 15.', 'tool_calls': None}]

In [18]:
agent.memory.reset()

In [19]:
agent.invoke("What is 2 to the power of 3")

{'role': 'assistant',
 'content': '2 to the power of 3 is 8.',
 'tool_calls': None}

In [20]:
agent.memory.get_messages()

[{'role': 'user', 'content': 'What is 2 to the power of 3', 'tool_calls': {}},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_10crsBFvUJC51lzURp3eaklL', function=Function(arguments='{"base":2,"exponent":3}', name='power'), type='function')]},
 {'role': 'tool',
  'content': '8',
  'tool_call_id': 'call_10crsBFvUJC51lzURp3eaklL'},
 {'role': 'assistant',
  'content': '2 to the power of 3 is 8.',
  'tool_calls': None}]

In [21]:
agent.invoke("What is 3 to the power of (2 to the power of 2)?")

{'role': 'assistant',
 'content': '3 to the power of (2 to the power of 2) is 81.',
 'tool_calls': None}

In [22]:
agent.memory.get_messages()

[{'role': 'user', 'content': 'What is 2 to the power of 3', 'tool_calls': {}},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_10crsBFvUJC51lzURp3eaklL', function=Function(arguments='{"base":2,"exponent":3}', name='power'), type='function')]},
 {'role': 'tool',
  'content': '8',
  'tool_call_id': 'call_10crsBFvUJC51lzURp3eaklL'},
 {'role': 'assistant',
  'content': '2 to the power of 3 is 8.',
  'tool_calls': None},
 {'role': 'user',
  'content': 'What is 3 to the power of (2 to the power of 2)?',
  'tool_calls': {}},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_W8r2p2fcVzvmqF8try65Y13O', function=Function(arguments='{"base": 2, "exponent": 2}', name='power'), type='function'),
   ChatCompletionMessageFunctionToolCall(id='call_ZaXucnougpu17Ec2I63VvpBa', function=Function(arguments='{"base": 3, "exponent": 0}', name='power'), type='function')]},
 {'role': 'tool',
  'con

## 6. 實驗

現在你已經理解它的運作方式，可以嘗試做一些新的實驗。

- 嘗試新的工具定義或工具說明。
- 當工具參數不完整或型別不正確時，會發生什麼事？
- 試著存取記憶內容來檢查工具呼叫流程（`agent.memory`），而不是只閱讀輸出。
- 你還可以嘗試什麼？
